[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/40.%20The%20Port%20Capacity%20%26%20Expansion%20Timing%20Problem/P40-Tier-5_executed.ipynb)

---


# 40. The Port Capacity and Expansion Timing Problem
## Tier 5: Integrated Digital-Twin-Inspired Scenario Simulation

### Goal
This notebook evaluates port expansion scenarios with a small, time-stepped simulation of interacting port subsystems. It illustrates how a digital-twin-style model can connect berth, yard, gate, and intermodal indicators, track operational variation over time, and compare investment scenarios using simulated congestion costs.

The implementation is a teaching model, not a production digital twin. It does not connect to sensors, ingest live operational data, forecast demand, or represent detailed equipment physics. Its results are conditional on the stated assumptions and should not be interpreted as empirical forecasts.

### Inputs, outputs, and scope
- **Inputs:** The baseline port parameters, four predefined investment scenarios, a simulation duration, and a fixed random seed.
- **Outputs:** Daily simulated indicators, scenario summaries, a financial comparison, and visualizations.
- **Scope:** The model represents berth utilization, yard utilization, gate throughput, intermodal coordination, a composite system-efficiency index, and a stylized daily congestion cost. It omits demand growth, detailed vessel and truck queues, construction schedules, operating costs, taxes, financing, discounting, and uncertainty in investment costs.

### Model assumptions
- Each simulation day receives one random operational multiplier drawn from a normal distribution with mean 1.0 and standard deviation 0.1. The same multiplier affects berth utilization, yard utilization, gate throughput, and intermodal utilization on that day.
- Utilization values are bounded to keep them within specified operational ranges. Gate throughput is represented as trucks per hour; the gate-efficiency contribution to the composite index is capped at 1.0.
- The average gate queue is held at its baseline value. Consequently, the gate-queue component of congestion cost does not respond to the expansion scenarios.
- The system-efficiency index is a constructed weighted score. Its berth and yard terms measure unused capacity, not conventional productivity; it is not an observed port KPI.
- The annual capacity estimate changes only with berth additions. The stated integrated-scenario investment cost is treated as a bundled scenario cost, rather than recalculated from the costs of its individual components.
- The financial comparison assumes that annual congestion savings persist for five years and uses no discount rate. The resulting measure is therefore an **undiscounted five-year net benefit**, not a discounted net present value.

### Simulation logic
A scenario first changes the subsystem parameters, then advances the port state one day at a time. The model records daily KPIs and summarizes them after the simulation. Each scenario uses the same seeded sequence of daily random multipliers, so scenario comparisons are not driven by different random draws.

Let $s$ denote a scenario and $t$ a simulation day. The daily random multiplier is $Z_t \sim \mathcal{N}(1, 0.1^2)$. For example, berth utilization follows
$$
u^{B}_{s,t} = \operatorname{clip}\!\left(u^{B}_{s,t-1} Z_t,\, 0.3,\, 0.98\right),
$$
and yard utilization follows the same update with bounds 0.3 and 0.95. Gate throughput is updated as $q_{s,t}=\lfloor q_{s,t-1}Z_t\rfloor$, while intermodal utilization uses bounds 0.4 and 0.9. The functions `_update_daily_operations` and `_calculate_kpis` implement these rules.

The daily composite index and congestion cost are
$$
\begin{aligned}
E_{s,t} ={}& 0.30(1-u^{B}_{s,t}) + 0.25(1-u^{Y}_{s,t}) \\
&+ 0.25\min\!\left(\frac{q_{s,t}}{Q},1\right)
+ 0.20c^{I}_{s,t}, \\
C_{s,t} ={}& 10{,}000\max(0,u^{B}_{s,t}-0.8)
+ 8{,}000\max(0,u^{Y}_{s,t}-0.8) \\
&+ 500\max(0,L-30).
\end{aligned}
$$
Here, $u^{B}_{s,t}$ and $u^{Y}_{s,t}$ are berth and yard utilization, $q_{s,t}$ is gate throughput, $Q$ is nominal gate capacity, $c^{I}_{s,t}$ is intermodal coordination efficiency, and $L$ is the fixed average gate queue. The coefficients define the illustrative cost scale in dollars per day; they are assumptions, not estimates from observed data.

For a scenario with average daily congestion cost $\bar C_s$ and investment $I_s$ in millions of dollars, annual savings relative to the simulated baseline are
$$
S_s = \frac{365(\bar C_{\mathrm{baseline}}-\bar C_s)}{1{,}000{,}000},
$$
and the undiscounted five-year net benefit is
$$
B_s = 5S_s-I_s.
$$
The code evaluates four scenarios over a fixed number of days, so its simulation work grows in proportion to the number of scenarios times the number of simulated days. The formulas and code describe a stylized scenario-comparison method; they do not implement live digital-twin synchronization or predictive analytics.

### How to interpret the results
A higher composite index is better only under the index definition above; it is not a general measure of port productivity. Lower simulated congestion cost and higher five-year net benefit are favorable under the financial assumptions. A scenario ranking is conditional on the model structure, fixed baseline data, chosen cost coefficients, and unmodeled factors. It is a planning illustration, not a stand-alone investment recommendation.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# A fixed seed makes the illustrative simulation reproducible.
np.random.seed(42)

plt.style.use("default")
sns.set_palette("husl")

print("Digital-twin-inspired port scenario simulation initialized.")

In [ ]:
@dataclass
class DigitalTwinParameters:
    """Baseline parameters for the illustrative port simulation."""

    current_capacity: float = 3.2  # Million TEU per year
    current_berths: int = 4
    current_cranes: int = 12
    gate_capacity: int = 200  # Trucks per hour
    yard_capacity: float = 10000  # TEU


class PortDigitalTwin:
    """Time-stepped scenario model of connected port subsystems."""

    def __init__(self, params: DigitalTwinParameters):
        self.params = params
        self.berth_operations = self._initialize_berth_system()
        self.yard_management = self._initialize_yard_system()
        self.gate_processing = self._initialize_gate_system()
        self.intermodal_integration = self._initialize_intermodal_system()

    def _initialize_berth_system(self) -> Dict:
        """Set baseline berth state and operating indicators."""
        return {
            "berths": self.params.current_berths,
            "cranes": self.params.current_cranes,
            "utilization": 0.873,
            "peak_utilization": 0.942,
            "vessel_queue": [],
            "productivity": 25.0,  # Moves per hour
        }

    def _initialize_yard_system(self) -> Dict:
        """Set baseline yard capacity, occupancy, and equipment."""
        return {
            "capacity": self.params.yard_capacity,
            "utilization": 0.784,
            "peak_utilization": 0.917,
            "containers": int(self.params.yard_capacity * 0.784),
            "density": 0.784,
            "equipment": 40,  # Yard cranes
        }

    def _initialize_gate_system(self) -> Dict:
        """Set baseline gate throughput, queue, and service indicators."""
        return {
            "capacity": self.params.gate_capacity,
            "throughput": 167,  # Trucks per hour
            "peak_throughput": 234,
            "queue_length": 45,  # Average trucks waiting
            "processing_time": 12,  # Minutes per truck
            "efficiency": 0.85,
        }

    def _initialize_intermodal_system(self) -> Dict:
        """Set baseline intermodal capacities and coordination."""
        return {
            "rail_capacity": 500,  # TEU per day
            "truck_capacity": 800,  # TEU per day
            "barge_capacity": 200,  # TEU per day
            "utilization": 0.721,
            "coordination_efficiency": 0.81,
        }

    def simulate_expansion_scenario(
        self,
        scenario_name: str,
        berth_addition: int = 0,
        automation_investment: float = 0,
        yard_expansion: float = 0,
        duration_days: int = 365,
    ) -> Dict:
        """Apply one scenario, simulate daily operations, and restore state."""
        if duration_days <= 0:
            raise ValueError("duration_days must be a positive integer.")
        if berth_addition < 0 or automation_investment < 0 or yard_expansion < 0:
            raise ValueError("Expansion inputs must be non-negative.")

        initial_state = self._capture_system_state()
        self._apply_expansion_changes(
            berth_addition,
            automation_investment,
            yard_expansion,
        )

        simulation_results = []
        try:
            for day in range(duration_days):
                self._update_daily_operations()
                daily_kpis = self._calculate_kpis()
                daily_kpis["day"] = day
                daily_kpis["scenario"] = scenario_name
                simulation_results.append(daily_kpis)
        finally:
            # Each scenario starts from the same port state, not the prior scenario's state.
            self._restore_system_state(initial_state)

        return {
            "daily_results": simulation_results,
            "summary": self._analyze_simulation_results(simulation_results),
            "scenario_name": scenario_name,
        }

    def _apply_expansion_changes(
        self,
        berth_addition: int,
        automation_investment: float,
        yard_expansion: float,
    ) -> None:
        """Translate each scenario's investment choices into model parameters."""
        if berth_addition > 0:
            self.berth_operations["berths"] += berth_addition
            self.berth_operations["cranes"] += berth_addition * 3
            self.berth_operations["utilization"] *= 0.7
            self.berth_operations["peak_utilization"] *= 0.75

        if automation_investment > 0:
            automation_level = automation_investment / 1000
            self.berth_operations["productivity"] *= 1 + 0.3 * automation_level
            self.yard_management["equipment"] = int(
                self.yard_management["equipment"] * (1 + 0.2 * automation_level)
            )
            self.gate_processing["efficiency"] *= 1 + 0.35 * automation_level
            self.intermodal_integration["coordination_efficiency"] *= (
                1 + 0.25 * automation_level
            )

        if yard_expansion > 0:
            self.yard_management["capacity"] *= 1 + yard_expansion
            self.yard_management["utilization"] *= 0.8
            self.yard_management["peak_utilization"] *= 0.85

    def _update_daily_operations(self) -> None:
        """Apply one shared daily shock to linked subsystem indicators."""
        daily_variation = np.random.normal(1.0, 0.1)

        self.berth_operations["utilization"] = np.clip(
            self.berth_operations["utilization"] * daily_variation,
            0.3,
            0.98,
        )
        self.yard_management["utilization"] = np.clip(
            self.yard_management["utilization"] * daily_variation,
            0.3,
            0.95,
        )
        self.gate_processing["throughput"] = int(
            self.gate_processing["throughput"] * daily_variation
        )
        self.intermodal_integration["utilization"] = np.clip(
            self.intermodal_integration["utilization"] * daily_variation,
            0.4,
            0.9,
        )

    def _calculate_kpis(self) -> Dict:
        """Calculate the defined composite index, cost, and capacity estimate."""
        berth_headroom = 1.0 - self.berth_operations["utilization"]
        yard_headroom = 1.0 - self.yard_management["utilization"]
        gate_efficiency = min(
            self.gate_processing["throughput"] / self.params.gate_capacity,
            1.0,
        )
        intermodal_efficiency = self.intermodal_integration[
            "coordination_efficiency"
        ]

        system_efficiency = (
            berth_headroom * 0.30
            + yard_headroom * 0.25
            + gate_efficiency * 0.25
            + intermodal_efficiency * 0.20
        )

        congestion_cost = (
            max(0, self.berth_operations["utilization"] - 0.8) * 10000
            + max(0, self.yard_management["utilization"] - 0.8) * 8000
            + max(0, self.gate_processing["queue_length"] - 30) * 500
        )

        total_capacity = self.params.current_capacity * (
            1
            + 0.4
            * (self.berth_operations["berths"] - self.params.current_berths)
        )

        return {
            "system_efficiency": system_efficiency,
            "berth_utilization": self.berth_operations["utilization"],
            "yard_utilization": self.yard_management["utilization"],
            "gate_throughput": self.gate_processing["throughput"],
            "congestion_cost": congestion_cost,
            "total_capacity": total_capacity,
        }

    def _capture_system_state(self) -> Dict:
        """Take a snapshot so scenario effects do not carry into another run."""
        return {
            "berth_operations": self.berth_operations.copy(),
            "yard_management": self.yard_management.copy(),
            "gate_processing": self.gate_processing.copy(),
            "intermodal_integration": self.intermodal_integration.copy(),
        }

    def _restore_system_state(self, state: Dict) -> None:
        """Restore each subsystem to its captured pre-scenario state."""
        self.berth_operations = state["berth_operations"].copy()
        self.yard_management = state["yard_management"].copy()
        self.gate_processing = state["gate_processing"].copy()
        self.intermodal_integration = state["intermodal_integration"].copy()

    def _analyze_simulation_results(self, results: List[Dict]) -> Dict:
        """Summarize daily outputs without imposing a financial baseline."""
        if not results:
            raise ValueError("At least one daily result is required.")

        df = pd.DataFrame(results)
        return {
            "avg_system_efficiency": df["system_efficiency"].mean(),
            "avg_congestion_cost": df["congestion_cost"].mean(),
            "peak_berth_utilization": df["berth_utilization"].max(),
            "avg_gate_throughput": df["gate_throughput"].mean(),
            "total_congestion_cost": df["congestion_cost"].sum(),
            "avg_total_capacity": df["total_capacity"].mean(),
        }


print("Port subsystem model and daily simulation are ready.")

In [ ]:
print("\n" + "=" * 80)
print("PORT CAPACITY EXPANSION SCENARIO SIMULATION")
print("=" * 80)

dt_params = DigitalTwinParameters()
digital_twin = PortDigitalTwin(dt_params)

initial_kpis = digital_twin._calculate_kpis()
print("\nINITIAL MODEL STATE")
print(f"Composite system-efficiency index: {initial_kpis['system_efficiency']:.1%}")
print(f"Berth utilization: {initial_kpis['berth_utilization']:.1%}")
print(f"Yard utilization: {initial_kpis['yard_utilization']:.1%}")
print(f"Gate throughput: {initial_kpis['gate_throughput']} trucks/hour")
print(f"Daily congestion cost: ${initial_kpis['congestion_cost']:,.0f}")

scenarios = {
    "Baseline": {
        "berth_addition": 0,
        "automation_investment": 0,
        "yard_expansion": 0,
        "investment_cost": 0,
    },
    "Berth Addition": {
        "berth_addition": 2,
        "automation_investment": 0,
        "yard_expansion": 0,
        "investment_cost": 480,  # Millions of dollars
    },
    "Automation Investment": {
        "berth_addition": 0,
        "automation_investment": 340,
        "yard_expansion": 0,
        "investment_cost": 340,  # Millions of dollars
    },
    "Integrated Approach": {
        "berth_addition": 2,
        "automation_investment": 340,
        "yard_expansion": 0.2,
        "investment_cost": 720,  # Bundled scenario cost, in millions of dollars
    },
}

print("\nSCENARIO INPUTS")
for name, config in scenarios.items():
    print(f"\n{name}:")
    print(f"  Investment: ${config['investment_cost']}M")
    print(f"  Berth additions: {config['berth_addition']}")
    print(f"  Automation investment: ${config['automation_investment']}M")
    print(f"  Yard expansion: {config['yard_expansion']:.0%}")

print("\n" + "=" * 60)
print("RUNNING DAILY SIMULATIONS")
print("=" * 60)

simulation_results = {}
common_random_state = np.random.get_state()

for scenario_name, config in scenarios.items():
    # Reusing the seeded random sequence isolates scenario effects from random-draw differences.
    np.random.set_state(common_random_state)
    result = digital_twin.simulate_expansion_scenario(
        scenario_name=scenario_name,
        berth_addition=config["berth_addition"],
        automation_investment=config["automation_investment"],
        yard_expansion=config["yard_expansion"],
        duration_days=365,
    )
    simulation_results[scenario_name] = result
    summary = result["summary"]
    print(f"\n{scenario_name}")
    print(f"  Average composite index: {summary['avg_system_efficiency']:.1%}")
    print(f"  Average daily congestion cost: ${summary['avg_congestion_cost']:,.0f}")
    print(f"  Peak berth utilization: {summary['peak_berth_utilization']:.1%}")
    print(f"  Average gate throughput: {summary['avg_gate_throughput']:.1f} trucks/hour")

In [ ]:
print("\n" + "=" * 80)
print("SCENARIO COMPARISON AND FINANCIAL ANALYSIS")
print("=" * 80)

baseline_summary = simulation_results["Baseline"]["summary"]
comparison_data = []

for scenario_name, result in simulation_results.items():
    summary = result["summary"]
    config = scenarios[scenario_name]

    annual_savings_m = (
        baseline_summary["avg_congestion_cost"] - summary["avg_congestion_cost"]
    ) * 365 / 1_000_000
    five_year_net_benefit_m = annual_savings_m * 5 - config["investment_cost"]
    roi_percent = (
        five_year_net_benefit_m / config["investment_cost"] * 100
        if config["investment_cost"] > 0
        else None
    )
    efficiency_change_percent = (
        (summary["avg_system_efficiency"]
         / baseline_summary["avg_system_efficiency"] - 1)
        * 100
        if baseline_summary["avg_system_efficiency"] != 0
        else None
    )

    comparison_data.append(
        {
            "Scenario": scenario_name,
            "Investment ($M)": config["investment_cost"],
            "Average Composite Index": summary["avg_system_efficiency"],
            "Peak Berth Utilization": summary["peak_berth_utilization"],
            "Annual Congestion Savings ($M)": annual_savings_m,
            "5-Year Net Benefit ($M)": five_year_net_benefit_m,
            "ROI (%)": roi_percent,
            "Index Change vs. Baseline (%)": efficiency_change_percent,
        }
    )

comparison_df = pd.DataFrame(comparison_data)
display_df = comparison_df.copy()
for column in ["Average Composite Index", "Peak Berth Utilization"]:
    display_df[column] = display_df[column].map(lambda value: f"{value:.1%}")
for column in [
    "Investment ($M)",
    "Annual Congestion Savings ($M)",
    "5-Year Net Benefit ($M)",
]:
    display_df[column] = display_df[column].map(lambda value: f"{value:,.2f}")
display_df["ROI (%)"] = display_df["ROI (%)"].map(
    lambda value: "N/A" if pd.isna(value) else f"{value:.1f}"
)
display_df["Index Change vs. Baseline (%)"] = display_df[
    "Index Change vs. Baseline (%)"
].map(lambda value: "N/A" if pd.isna(value) else f"{value:+.1f}%")
display(display_df)

maximum_net_benefit = comparison_df["5-Year Net Benefit ($M)"].max()
best_scenarios = comparison_df[
    comparison_df["5-Year Net Benefit ($M)"].apply(
        lambda value: np.isclose(value, maximum_net_benefit, rtol=1e-9, atol=1e-9)
    )
]
best_scenario = best_scenarios.iloc[0]

print("\nScenario(s) with the highest modeled five-year net benefit:")
for _, row in best_scenarios.iterrows():
    print(
        f"  {row['Scenario']}: "
        f"${row['5-Year Net Benefit ($M)']:.2f}M"
    )

print("\nINTERPRETIVE COMPARISONS")
for _, row in comparison_df.iterrows():
    print(
        f"  {row['Scenario']}: "
        f"index change {row['Index Change vs. Baseline (%)']:+.1f}%"
        if not pd.isna(row["Index Change vs. Baseline (%)"])
        else f"  {row['Scenario']}: index change not defined"
    )

print("\nThe financial ranking uses undiscounted savings under the stated model assumptions.")

In [ ]:
scenario_names = list(simulation_results.keys())
scenario_colors = {
    name: color
    for name, color in zip(
        scenario_names,
        ["lightcoral", "skyblue", "lightgreen", "gold"],
    )
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    "Port Expansion Scenario Simulation Results",
    fontsize=16,
    fontweight="bold",
)

# Compare the simulated composite index across scenarios.
efficiencies = [
    simulation_results[name]["summary"]["avg_system_efficiency"]
    for name in scenario_names
]
bars = axes[0, 0].bar(
    scenario_names,
    efficiencies,
    color=[scenario_colors[name] for name in scenario_names],
)
axes[0, 0].set_title("Average Composite System-Efficiency Index")
axes[0, 0].set_ylabel("Index value (unitless)")
axes[0, 0].tick_params(axis="x", rotation=35)
axes[0, 0].grid(True, axis="y", alpha=0.3)
for bar, value in zip(bars, efficiencies):
    axes[0, 0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{value:.1%}",
        ha="center",
        va="bottom",
    )

# Compare the average daily congestion cost in dollars.
congestion_costs = [
    simulation_results[name]["summary"]["avg_congestion_cost"]
    for name in scenario_names
]
bars = axes[0, 1].bar(
    scenario_names,
    congestion_costs,
    color=[scenario_colors[name] for name in scenario_names],
)
axes[0, 1].set_title("Average Daily Congestion Cost")
axes[0, 1].set_ylabel("Cost (USD per day)")
axes[0, 1].tick_params(axis="x", rotation=35)
axes[0, 1].grid(True, axis="y", alpha=0.3)
for bar, value in zip(bars, congestion_costs):
    axes[0, 1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(congestion_costs, default=0) * 0.02,
        f"${value:,.0f}",
        ha="center",
        va="bottom",
    )

# Show investment and the corresponding undiscounted five-year net benefit.
scenario_lookup = comparison_df.set_index("Scenario")
investments = [scenario_lookup.loc[name, "Investment ($M)"] for name in scenario_names]
net_benefits = [
    scenario_lookup.loc[name, "5-Year Net Benefit ($M)"]
    for name in scenario_names
]
axes[1, 0].scatter(
    investments,
    net_benefits,
    s=100,
    c=[scenario_colors[name] for name in scenario_names],
    edgecolors="black",
    alpha=0.8,
)
axes[1, 0].axhline(0, color="black", linewidth=1, linestyle="--")
axes[1, 0].set_title("Investment and Five-Year Net Benefit")
axes[1, 0].set_xlabel("Investment (millions of USD)")
axes[1, 0].set_ylabel("Undiscounted net benefit (millions of USD)")
axes[1, 0].grid(True, alpha=0.3)
for investment, net_benefit, name in zip(
    investments,
    net_benefits,
    scenario_names,
):
    axes[1, 0].annotate(
        name,
        (investment, net_benefit),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=9,
    )

# Plot sampled daily index values to show the simulated time paths.
sample_days = list(range(0, 365, 30))
for scenario_name, result in simulation_results.items():
    daily_results = result["daily_results"]
    sampled_results = [daily_results[day] for day in sample_days if day < len(daily_results)]
    axes[1, 1].plot(
        [row["day"] for row in sampled_results],
        [row["system_efficiency"] for row in sampled_results],
        marker="o",
        label=scenario_name,
        linewidth=2,
        color=scenario_colors[scenario_name],
    )
axes[1, 1].set_title("Composite Index over Sampled Simulation Days")
axes[1, 1].set_xlabel("Simulation day")
axes[1, 1].set_ylabel("Index value (unitless)")
axes[1, 1].legend(title="Scenario")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Scenario charts are ready for interpretation.")

In [ ]:
print("\n" + "=" * 80)
print("SCENARIO SUMMARY AND PLANNING NOTES")
print("=" * 80)

best_result = simulation_results[best_scenario["Scenario"]]
best_config = scenarios[best_scenario["Scenario"]]
baseline_result = simulation_results["Baseline"]["summary"]
best_summary = best_result["summary"]

print("\nHIGHEST MODELED FIVE-YEAR NET BENEFIT")
print(f"  Scenario: {best_scenario['Scenario']}")
print(f"  Investment: ${best_config['investment_cost']}M")
print(
    "  Undiscounted five-year net benefit: "
    f"${best_scenario['5-Year Net Benefit ($M)']:.2f}M"
)
if pd.isna(best_scenario["ROI (%)"]):
    print("  ROI: Not defined for a zero-investment scenario")
else:
    print(f"  ROI: {best_scenario['ROI (%)']:.1f}%")

print("\nSIMULATED PERFORMANCE RELATIVE TO THE BASELINE")
baseline_index = baseline_result["avg_system_efficiency"]
best_index = best_summary["avg_system_efficiency"]
if baseline_index != 0:
    index_change = (best_index / baseline_index - 1) * 100
    print(f"  Composite-index change: {index_change:+.1f}%")
else:
    print("  Composite-index change: not defined because the baseline index is zero")
print(
    "  Peak berth utilization: "
    f"{baseline_result['peak_berth_utilization']:.1%} to "
    f"{best_summary['peak_berth_utilization']:.1%}"
)
print(
    "  Average daily congestion cost: "
    f"${baseline_result['avg_congestion_cost']:,.0f} to "
    f"${best_summary['avg_congestion_cost']:,.0f}"
)

annual_savings_m = best_scenario["Annual Congestion Savings ($M)"]
if annual_savings_m > 0:
    payback_period = best_config["investment_cost"] / annual_savings_m
    print(f"  Simple payback from modeled congestion savings: {payback_period:.1f} years")
else:
    print("  Simple payback: not defined because modeled annual savings are not positive")

print("\nILLUSTRATIVE IMPLEMENTATION CONSIDERATIONS")
if best_scenario["Scenario"] == "Integrated Approach":
    print("  Coordinate berth, automation, and yard work through staged design and commissioning.")
elif best_scenario["Scenario"] == "Berth Addition":
    print("  Validate site, berth, and crane requirements before detailed design and commissioning.")
elif best_scenario["Scenario"] == "Automation Investment":
    print("  Assess technology, operational integration, staff training, and commissioning needs.")
else:
    print("  Use the baseline as a reference case; it includes no expansion investment.")

print("\nMODEL LIMITATIONS TO CONSIDER")
print("  - The modeled queue is fixed, so gate-queue congestion does not respond to investment.")
print("  - Daily variation is a simple shared random multiplier, not a calibrated demand process.")
print("  - The financial comparison excludes discounting, operating costs, and revenue changes.")
print("  - A scenario ranking is conditional on the model assumptions and is not a forecast.")

print("\n" + "=" * 80)
print("Scenario simulation and comparison are complete.")
print("=" * 80)